# obj-wm Playground

Load the v2 world model and a mini scene, try four different ego plans,
and see how the predicted reactions of nearby agents change.

**Kernel:** `.venv_sim` (python 3.10 with MetaDrive and ScenarioNet).  
**Data:** converted mini scenarios in `sim/scenarios/nuscenes_mini/`.  
Run `make setup` and `make quickstart` first if you haven't yet.

## 1. Configuration

In [ ]:
import os, sys
from pathlib import Path

# Point to the project root (wherever the notebook lives)
PROJECT_ROOT = Path(globals().get('__file__', '.')).resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')   # offscreen render

CHECKPOINT = PROJECT_ROOT / 'checkpoints/v2_d_rerun/v2_seed42_best.pth'
SCENARIO_DIR = PROJECT_ROOT / 'sim/scenarios/nuscenes_mini'
SCENE_IDX = 0      # integer index into the converted mini scenes (0 = first scene)
DEVICE = 'cpu'     # or 'cuda'

## 2. Load the world model

The `WorldModelPredictor` wraps the v2 scene transformer.
It takes all 35 candidate plans (7 speeds × 5 offsets) in one batch
and returns predicted trajectories for 16 nearby agents under **each plan**.
Agents the model cannot predict fall back to constant-velocity.

If the checkpoint is missing, run `make quickstart` to train a mini model,
or download the official weights (see the README).

In [ ]:
from sim.closed_loop import WorldModelPredictor, CVPredictor

if CHECKPOINT.exists():
    wm = WorldModelPredictor(str(CHECKPOINT), DEVICE)
    print(f'Loaded checkpoint: {CHECKPOINT.name}')
    print(f'Model predicts {len(wm.model.slots)} agents, {wm.model.K} modes')
else:
    print(f'Checkpoint not found: {CHECKPOINT}')
    print('Using CV predictor (no plan conditioning).')
    wm = CVPredictor()

## 3. Load a scene and warm up

The model needs 2 seconds of history (20 steps at 10 Hz).
We run the environment for 20 steps with the ego following its log
so the history buffer fills up before we start experimenting.

In [ ]:
from metadrive.envs.scenario_env import ScenarioEnv
from metadrive.scenario.utils import get_number_of_scenarios
from sim.closed_loop import HistoryBuffer, Scene, EGO_SIZE_ZOE, set_ego_to_log
from sim.adapter import add_connector_polygons, snapshot_from_env
from sim.planner import reference_from_scenario, generate_plans, SPEEDS, OFFSETS

n = get_number_of_scenarios(str(SCENARIO_DIR))
env = ScenarioEnv(dict(
    use_render=False, num_scenarios=n, data_directory=str(SCENARIO_DIR),
    horizon=10_000, log_level=50,
    crash_vehicle_done=False, crash_object_done=False, crash_human_done=False,
    vehicle_config=dict(vehicle_model='varying_dynamics',
                        width=EGO_SIZE_ZOE[0], length=EGO_SIZE_ZOE[1]),
    agent_configs={'default_agent': dict(
        use_special_color=True, spawn_lane_index=None,
        vehicle_model='varying_dynamics',
        width=EGO_SIZE_ZOE[0], length=EGO_SIZE_ZOE[1])},
))
env.reset(seed=SCENE_IDX)

scenario = env.engine.data_manager.current_scenario
features = add_connector_polygons(scenario['map_features'], scenario['metadata']['map'])
ref = reference_from_scenario(scenario)
scene_id = scenario['metadata'].get('scenario_id', f'idx-{SCENE_IDX}')
print(f'Scene: {scene_id}')

# Warm-up: follow the log for 20 steps (2 s) so the history buffer has real data
WARMUP = 20
buffer = HistoryBuffer()
buffer.push(snapshot_from_env(env, None))
for step in range(WARMUP):
    env.step([0.0, 0.0])
    set_ego_to_log(env, scenario, step + 1)
    buffer.push(snapshot_from_env(env, None))

a = env.agent
state = (float(a.position[0]), float(a.position[1]),
         float(a.heading_theta), float(a.speed))
print(f'Ego at step {WARMUP}: x={state[0]:.1f} y={state[1]:.1f} '
      f'yaw={state[2]:.2f} rad  speed={state[3]:.1f} m/s')

## 4. Generate plans and run predictions

We generate all 35 candidate plans and call `predict()` once
(the model runs the whole batch in one forward pass).
Then we highlight 4 plans:

| Plan | Speed target | Lateral offset |
|---|---|---|
| Brake | 0 m/s | 0 m |
| Keep speed | 4 m/s | 0 m |
| Accelerate | 8 m/s | 0 m |
| Offset left | 4 m/s | +0.75 m |

The plan is encoded as `ego_future`: the ego's intended 6 s trajectory
in the agent's local frame.  The model sees this as extra tokens
alongside the agent and lane tokens.

In [ ]:
import numpy as np

scene = Scene(buffer, features, state, WARMUP)
sparse, dense, meta = generate_plans(ref, state)

# Predict under all 35 plans at once
paths, probs, agent_ids = wm.predict(scene, sparse)
# paths: (P=35, A=16, K, 12, 2) world xy at t+0.5..6 s
# probs: (P, A, K)  per-plan, per-agent, per-mode probabilities
print(f'Prediction shape: paths {paths.shape}, probs {probs.shape}')
print(f'Agents predicted: {agent_ids}')

# Select 4 highlight plans by (speed, offset)
HIGHLIGHT = [
    (0.0,  0.0,  'brake',         '#e87628'),
    (4.0,  0.0,  'keep speed',    '#1a7a3f'),
    (8.0,  0.0,  'accelerate',    '#2468d6'),
    (4.0,  0.75, 'offset left',   '#9640c8'),
]
plan_indices = {}
for sp, off, label, color in HIGHLIGHT:
    idx = meta.index((sp, off))
    plan_indices[label] = idx
    print(f'  plan {idx:2d}: speed={sp} m/s  offset={off:+.2f} m  → "{label}"')

## 5. Find the most interesting nearby agent

We pick the nearest moving agent that the WM actually predicts
(falls back to CV for agents further away or not in the model's 16 slots).

In [ ]:
from sim.closed_loop import agent_states

agents = agent_states(scene)   # list of (oid, x, y, yaw, v, type)

# Find the nearest agent that is moving and that the WM predicted
ex, ey = state[0], state[1]
best_oid, best_dist = None, np.inf
for oid, x, y, yaw, v, atype in agents:
    if oid == 0:          # skip ego
        continue
    d = np.hypot(x - ex, y - ey)
    if d < best_dist and v > 0.5:
        best_dist, best_oid = d, oid

if best_oid is None:
    best_oid = agent_ids[0] if len(agent_ids) else None

if best_oid is not None and best_oid in agent_ids:
    slot = list(agent_ids).index(best_oid)
    print(f'Focus agent: id={best_oid}  dist={best_dist:.1f} m  slot {slot}')
else:
    slot = 0
    print(f'Focus agent: slot 0 (id={agent_ids[0] if len(agent_ids) else "n/a"})')

## 6. Plot: predicted reactions under 4 ego plans

Each panel shows the top-down view at the warm-up step:
- **Cyan line**: the ego's candidate plan.
- **Coloured lines**: the 6 predicted modes for the focus agent (opacity ∝ probability).
- **Grey lines**: constant-velocity reference for the same agent.

If the model is plan-conditioned, you should see the focus agent predict
differently under brake vs. accelerate — especially if that agent is
directly behind the ego.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

cv_pred = CVPredictor()
cv_paths, cv_probs, _ = cv_pred.predict(scene, sparse)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharex=True, sharey=True)
fig.suptitle(f'WM predictions under 4 ego plans — {scene_id} (step {WARMUP})',
             fontsize=12, y=1.01)

AGENT_COLORS = plt.cm.Set1.colors

for ax, (sp, off, label, ecolor) in zip(axes, HIGHLIGHT):
    p_idx = plan_indices[label]

    # ego plan
    plan_xy = sparse[p_idx, :, :2]          # (12, 2) world
    ax.plot([state[0]] + list(plan_xy[:, 0]),
            [state[1]] + list(plan_xy[:, 1]),
            color='#00aaaa', lw=2, zorder=5, label='ego plan')
    ax.plot(state[0], state[1], 'o', color='#00aaaa', ms=6, zorder=6)

    # other agents (all, grey)
    for oid, ax_, ay, ayaw, av, _ in agents:
        ax.plot(ax_, ay, 's', color='#aaaaaa', ms=4, zorder=2)

    # CV reference (grey dashed)
    for a_slot in range(cv_paths.shape[1]):
        traj = cv_paths[p_idx, a_slot, 0]   # K=1 for CV
        ax.plot(traj[:, 0], traj[:, 1], '--', color='#cccccc', lw=0.8, zorder=3)

    # WM predictions for the focus agent (all 6 modes)
    if slot < paths.shape[1]:
        K = paths.shape[2]
        for k in range(K):
            traj = paths[p_idx, slot, k]    # (12, 2)
            alpha = max(0.15, float(probs[p_idx, slot, k]))
            ax.plot(traj[:, 0], traj[:, 1],
                    color=AGENT_COLORS[slot % len(AGENT_COLORS)],
                    lw=1.2, alpha=alpha, zorder=4)

    ax.set_title(f'{label}\n(speed={sp} m/s, offset={off:+.2f} m)', fontsize=9)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x [m]')

axes[0].set_ylabel('y [m]')
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.savefig('outputs/playground_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/playground_predictions.png')

## 7. Inspect the probability shift

How much does the most-likely mode's probability change between plans?
A large shift means the model is reacting to our plan.
The v2 model's reaction is weaker than real drivers' (see the README, v3 ablations).

In [ ]:
print(f'Focus agent (slot {slot}) — top-mode probability and speed at t+2 s:')
print(f'  {"plan":<14s}  top-p   top-mode speed at t+2 s [m/s]')
for sp, off, label, _ in HIGHLIGHT:
    p_idx = plan_indices[label]
    if slot >= paths.shape[1]:
        print(f'  {label:<14s}  (agent not in prediction slots)')
        continue
    pr = probs[p_idx, slot]              # (K,)
    top_k = int(np.argmax(pr))
    top_p = float(pr[top_k])
    # speed ≈ displacement / 0.5 s at the t+2 s point (step 4)
    traj = paths[p_idx, slot, top_k]    # (12, 2)
    if traj.shape[0] >= 4:
        d = np.linalg.norm(traj[4] - traj[3])
        est_speed = d / 0.5
    else:
        est_speed = float('nan')
    print(f'  {label:<14s}  {top_p:.3f}  {est_speed:.1f}')

## 8. Try your own plan

Edit `MY_SPEED` and `MY_OFFSET` and re-run this cell.
Valid speeds: 0, 2, 4, 6, 8, 10, 12 m/s.  Valid offsets: -1.5, -0.75, 0, 0.75, 1.5 m.

In [ ]:
MY_SPEED  = 2.0    # m/s target
MY_OFFSET = 0.0    # m left of reference

try:
    my_idx = meta.index((MY_SPEED, MY_OFFSET))
except ValueError:
    print(f'({MY_SPEED}, {MY_OFFSET}) not in the grid.  Valid speeds: {list(SPEEDS)}, offsets: {list(OFFSETS)}')
    my_idx = None

if my_idx is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    plan_xy = sparse[my_idx, :, :2]
    ax.plot([state[0]] + list(plan_xy[:, 0]),
            [state[1]] + list(plan_xy[:, 1]),
            color='#00aaaa', lw=2, label='ego plan')
    ax.plot(state[0], state[1], 'o', color='#00aaaa', ms=6)
    for oid, ax_, ay, ayaw, av, _ in agents:
        ax.plot(ax_, ay, 's', color='#aaaaaa', ms=4)
    if slot < paths.shape[1]:
        K = paths.shape[2]
        for k in range(K):
            traj = paths[my_idx, slot, k]
            alpha = max(0.15, float(probs[my_idx, slot, k]))
            ax.plot(traj[:, 0], traj[:, 1],
                    color=AGENT_COLORS[slot % len(AGENT_COLORS)],
                    lw=1.5, alpha=alpha)
    ax.set_title(f'speed={MY_SPEED} m/s  offset={MY_OFFSET:+.2f} m')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.legend(); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    plt.tight_layout(); plt.show()

## Clean up

In [ ]:
env.close()
print('Environment closed.')